In [ ]:
!pip install -q "transformers==4.40.2" "huggingface_hub==0.23.4"

In [ ]:
import os, json, time, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import f1_score, accuracy_score

# Patch broken chat-template lookup in newer transformers (crashes on non-chat models)
import transformers.utils.hub as _hub
_hub.list_repo_templates = lambda *a, **kw: []

from transformers import BlipProcessor, BlipForImageTextRetrieval

In [ ]:
BASE_INPUT   = "/kaggle/input/datasets/youssefelghandour11/bigdataset/kaggle_dataset_full"
IMAGES_ROOT  = f"{BASE_INPUT}/images"
WORKING_DIR  = "/kaggle/working"

TRAIN_LABELS = f"{BASE_INPUT}/merged_balanced/train.json"
TRAIN_META   = f"{BASE_INPUT}/metadata/train.json"

CKPT_DIR     = f"{WORKING_DIR}/blip_itm_finetuned_v3/checkpoints"
BEST_DIR     = f"{WORKING_DIR}/blip_itm_finetuned_v3/best"
HISTORY_JSON = f"{WORKING_DIR}/blip_itm_finetuned_v3/training_history.json"

MODEL_NAME   = "Salesforce/blip-image-text-matching-base"

BATCH_SIZE      = 32
NUM_WORKERS     = 4
MAX_EPOCHS      = 30
PATIENCE        = 5
LR_HEAD         = 1e-5
LR_ENCODER      = 5e-6
UNFREEZE_LAYERS = 4

## 1. Data Loading

In [ ]:
def build_image_path(raw_path: str) -> str:
    stripped = raw_path.removeprefix("visual_news/")
    return os.path.join(IMAGES_ROOT, stripped)


def load_split(labels_path: str, meta_path: str, split_name: str):
    with open(labels_path, "r") as f:
        annotations = json.load(f)["annotations"]  # list of {id, image_id, falsified, ...}

    with open(meta_path, "r") as f:
        metadata = json.load(f)

    samples      = []
    missing_meta = 0
    missing_file = 0

    for ann in annotations:
        img_id = ann["image_id"]
        key    = str(img_id)

        if key not in metadata:
            missing_meta += 1
            continue

        meta     = metadata[key]
        img_path = build_image_path(meta["image_path"])
        caption  = meta.get("caption", "")

        if not os.path.exists(img_path):
            missing_file += 1
            continue

        samples.append({
            "image_id":   img_id,
            "image_path": img_path,
            "caption":    caption,
            "label":      int(ann["falsified"]),
        })

    print(f"[{split_name}] loaded={len(samples)} | "
          f"missing_meta={missing_meta} | missing_file={missing_file}")
    return samples

## 2. Dataset

In [ ]:
class ITMDataset(Dataset):
    def __init__(self, samples, processor, max_text_len=128):
        self.samples      = samples
        self.processor    = processor
        self.max_text_len = max_text_len
        self.corrupt_count = 0

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            image = Image.open(s["image_path"]).convert("RGB")
        except (UnidentifiedImageError, OSError):
            self.corrupt_count += 1
            image = Image.new("RGB", (384, 384))

        encoding = self.processor(
            images=image,
            text=s["caption"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"]     = torch.tensor(s["label"], dtype=torch.long)
        item["sample_idx"] = torch.tensor(idx, dtype=torch.long)
        return item


def make_loader(samples, processor, shuffle=True, batch_size=BATCH_SIZE):
    ds = ITMDataset(samples, processor)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    ), ds

## 3. Model Setup

In [ ]:
def freeze_model(model):
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_last_n_layers(encoder, n):
    all_layers = list(encoder.encoder.layer)
    for layer in all_layers[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


def build_model_and_optimizer():
    # Load fresh from pretrained — no checkpoint
    model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME)

    freeze_model(model)

    # Unfreeze last 4 layers of both encoders from the start
    unfreeze_last_n_layers(model.vision_model, UNFREEZE_LAYERS)
    unfreeze_last_n_layers(model.text_encoder, UNFREEZE_LAYERS)

    for name, p in model.named_parameters():
        if any(k in name for k in ["itm_head", "vision_proj", "text_proj"]):
            p.requires_grad = True

    head_params    = [p for n, p in model.named_parameters()
                      if p.requires_grad and
                      any(k in n for k in ["itm_head", "vision_proj", "text_proj"])]
    encoder_params = [p for n, p in model.named_parameters()
                      if p.requires_grad and
                      not any(k in n for k in ["itm_head", "vision_proj", "text_proj"])]

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    # 3 effective groups:
    #   head_params    -> LR 1e-5
    #   encoder_params -> LR 5e-6  (last 4 layers of both encoders)
    #   everything else is frozen (no grad), so not included
    optimizer = torch.optim.AdamW([
        {"params": head_params,    "lr": LR_HEAD},
        {"params": encoder_params, "lr": LR_ENCODER},
    ], lr=LR_ENCODER, weight_decay=1e-4)

    return model, optimizer

## 4. Training / Eval Utils

In [ ]:
def run_epoch(model, loader, optimizer, scaler, device, is_train):
    model.train() if is_train else model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss, all_preds, all_labels = 0.0, [], []
    first_batch = True

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            pixel_values   = batch["pixel_values"].to(device)
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            with autocast():
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_itm_head=True,
                )
                loss = criterion(outputs.itm_score, labels)

            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * labels.size(0)
            preds = outputs.itm_score.argmax(dim=-1).detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            if first_batch and is_train:
                first_batch = False
                for i in range(torch.cuda.device_count()):
                    mem = torch.cuda.memory_allocated(i) / 1e9
                    res = torch.cuda.memory_reserved(i) / 1e9
                    print(f"  GPU {i} after 1st batch: "
                          f"allocated={mem:.2f}GB  reserved={res:.2f}GB")

    n        = len(all_labels)
    avg_loss = total_loss / n
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    return avg_loss, acc, f1

## 5. Main Training Loop

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
def save_checkpoint(model, path: str):
    os.makedirs(path, exist_ok=True)
    inner = model.module if hasattr(model, "module") else model
    torch.save(copy.deepcopy(inner.state_dict()), os.path.join(path, "model.pt"))


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}  |  GPUs: {torch.cuda.device_count()}")

    # Full balanced dataset — all 71,072 samples, no hard-negative oversampling
    train_samples = load_split(TRAIN_LABELS, TRAIN_META, "TRAIN")

    processor = BlipProcessor.from_pretrained(MODEL_NAME)

    # Load fresh from pretrained; last 4 layers of both encoders unfrozen from epoch 1
    model, optimizer = build_model_and_optimizer()
    if torch.cuda.device_count() > 1:
        print(f"Wrapping model in DataParallel across "
              f"{torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model.to(device)

    scaler = GradScaler()

    history           = []
    best_train_f1     = -1.0
    epochs_no_improve = 0

    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs(BEST_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(HISTORY_JSON), exist_ok=True)

    print("\n" + "\u2550"*65)
    print("Starting training  [v3 — full dataset, 4 unfrozen layers]")
    print("\u2550"*65)

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        print(f"\n\u2500\u2500 Epoch {epoch}/{MAX_EPOCHS} "
              f"| train_set_size={len(train_samples)} \u2500\u2500")

        train_loader, train_ds = make_loader(
            train_samples, processor, shuffle=True)

        train_loss, train_acc, train_f1 = run_epoch(
            model, train_loader, optimizer, scaler, device, is_train=True)

        if train_ds.corrupt_count:
            print(f"  Corrupt images skipped this epoch: {train_ds.corrupt_count}")

        elapsed = time.time() - t0
        print(f"  train_loss={train_loss:.4f}  "
              f"train_acc={train_acc:.4f}  train_F1={train_f1:.4f}  "
              f"({elapsed:.0f}s)")

        # Save checkpoint for this epoch
        epoch_ckpt_path = os.path.join(CKPT_DIR, f"epoch_{epoch}")
        save_checkpoint(model, epoch_ckpt_path)
        print(f"  Checkpoint saved -> {epoch_ckpt_path}/model.pt")

        # Save best checkpoint separately
        if train_f1 > best_train_f1:
            best_train_f1 = train_f1
            save_checkpoint(model, BEST_DIR)
            print(f"  \u2713 New best train F1={best_train_f1:.4f} \u2014 best checkpoint saved -> {BEST_DIR}/model.pt")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        history.append({
            "epoch":       epoch,
            "train_loss":  round(train_loss, 6),
            "train_acc":   round(train_acc, 6),
            "train_f1":    round(train_f1, 6),
            "elapsed_sec": round(elapsed, 1),
        })

        with open(HISTORY_JSON, "w") as f:
            json.dump(history, f, indent=2)

        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

    print("\n" + "\u2550"*75)
    print(f"{'Epoch':>6} {'TrainLoss':>10} {'TrainAcc':>9} {'TrainF1':>8} {'Time':>7}")
    print("\u2500"*75)
    for r in history:
        print(f"{r['epoch']:>6} {r['train_loss']:>10.4f} "
              f"{r['train_acc']:>9.4f} {r['train_f1']:>8.4f} "
              f"{r['elapsed_sec']:>6.0f}s")
    print("\u2550"*75)
    print(f"Best train F1: {best_train_f1:.4f}")
    print(f"Best checkpoint: {BEST_DIR}/model.pt")
    print(f"Training history: {HISTORY_JSON}")


main()